# Univariate and Multivariate TimeSeries

This notebook shows how to use **univariate** (`TimeSeries`) and **multivariate** (`MultivariateTimeSeries`) time series from `timedatamodel`. `TimeSeries` handles a single value per timestamp with scalar metadata, while `MultivariateTimeSeries` handles multiple columns with list-based metadata and a 2D value array.

## 1. Univariate TimeSeries

A **univariate** TimeSeries has one value per timestamp. You create it by passing:
- `resolution` (frequency + timezone)
- `timestamps`: list of `datetime`
- `values`: list of `float | None` (or use `data` as a list of `DataPoint`)
- optional scalar metadata: `name`, `unit`, `data_type`, etc.

In [ ]:
from datetime import datetime, timedelta, timezone
from timedatamodel import TimeSeries, Resolution, DataPoint, Frequency, DataType

res = Resolution(Frequency.PT1H, "UTC")

base = datetime(2024, 6, 1, tzinfo=timezone.utc)
timestamps = [base + timedelta(hours=i) for i in range(6)]
values = [18.2, 19.1, 20.5, 22.0, 21.3, 19.8]

ts_uni = TimeSeries(res, timestamps=timestamps, values=values, name="temperature", unit="°C", data_type=DataType.ACTUAL)
ts_uni

### Univariate: properties and iteration

- `column_names` defaults to the `name` attribute (or `"value"`)
- Indexing and iteration yield **DataPoint** named tuples `(timestamp, value)`

In [ ]:
print("column_names:", ts_uni.column_names)

first = ts_uni[0]
print("\nFirst point:", first)
print("  .timestamp:", first.timestamp)
print("  .value:", first.value)

for i, dp in enumerate(ts_uni):
    if i < 2:
        print(dp)

### Univariate: NumPy and pandas

- `to_numpy()` returns a **1D** array.
- `to_pandas_dataframe()` returns a DataFrame with a single value column (and DatetimeIndex).

In [4]:
import numpy as np
import pandas as pd

arr = ts_uni.to_numpy()
print("to_numpy() shape:", arr.shape, "→ 1D array")

df_uni = ts_uni.to_pandas_dataframe()
print("\nto_pandas_dataframe() columns:", df_uni.columns.tolist())
df_uni

to_numpy() shape: (6,) → 1D array

to_pandas_dataframe() columns: ['temperature']


,temperature
timestamp,
2024-06-01 00:00:00+00:00,18.2
2024-06-01 01:00:00+00:00,19.1
2024-06-01 02:00:00+00:00,20.5
2024-06-01 03:00:00+00:00,22.0
2024-06-01 04:00:00+00:00,21.3
2024-06-01 05:00:00+00:00,19.8


## 2. MultivariateTimeSeries

A **MultivariateTimeSeries** has multiple values per timestamp (e.g. power and temperature at the same time). You create it by passing:
- `resolution`
- `timestamps`: list of `datetime`
- `values`: a **2D NumPy array** of shape `(n_timestamps, n_variables)`
- `names`: list of column names (one per variable)
- optional list metadata: `units`, `data_types`, etc. — each either length 1 (broadcast) or length `n_columns`

In [ ]:
from timedatamodel import MultivariateTimeSeries
import numpy as np

values_2d = np.array([
    [100.0, 18.2],
    [120.0, 19.1],
    [140.0, 20.5],
    [135.0, 22.0],
    [110.0, 21.3],
    [95.0, 19.8],
])

ts_multi = MultivariateTimeSeries(
    res,
    timestamps=timestamps,
    values=values_2d,
    names=["power_MW", "temperature_C"],
    data_types=[DataType.ACTUAL],
)
ts_multi

### Multivariate: properties and iteration

- `n_columns` is the number of value columns
- `column_names` are the names you passed (or `value_0`, `value_1`, ...)
- Indexing and iteration yield **tuples** `(timestamp, list_of_values)` — not `DataPoint`

In [ ]:
print("n_columns:", ts_multi.n_columns)
print("column_names:", ts_multi.column_names)

first = ts_multi[0]
print("\nFirst row:", first)
print("  timestamp:", first[0])
print("  values:  ", first[1])

print("\nFirst 2 rows:", [ts_multi[i] for i in range(2)])

### Multivariate: NumPy and pandas

- `to_numpy()` returns a **2D** array of shape `(n_timestamps, n_variables)`.
- `to_pandas_dataframe()` returns a DataFrame with **multiple columns** (one per variable).

In [8]:
arr_multi = ts_multi.to_numpy()
print("to_numpy() shape:", arr_multi.shape, "→ 2D array (rows=time, cols=variables)")

df_multi = ts_multi.to_pandas_dataframe()
print("\nto_pandas_dataframe() columns:", df_multi.columns.tolist())
df_multi

to_numpy() shape: (6, 2) → 2D array (rows=time, cols=variables)

to_pandas_dataframe() columns: ['power_MW', 'temperature_C']


,power_MW,temperature_C
timestamp,,
2024-06-01 00:00:00+00:00,100.0,18.2
2024-06-01 01:00:00+00:00,120.0,19.1
2024-06-01 02:00:00+00:00,140.0,20.5
2024-06-01 03:00:00+00:00,135.0,22.0
2024-06-01 04:00:00+00:00,110.0,21.3
2024-06-01 05:00:00+00:00,95.0,19.8


### Creating MultivariateTimeSeries from pandas

If your DataFrame has a DatetimeIndex and **multiple numeric columns**, `MultivariateTimeSeries.from_pandas()` produces a multivariate time series. Column names become `names`.

In [ ]:
ts_from_df = MultivariateTimeSeries.from_pandas(df_multi, resolution=res)
print("From pandas: n_columns =", ts_from_df.n_columns)
print("Column names:", ts_from_df.column_names)
print("Values match:", np.allclose(ts_from_df.to_numpy(), ts_multi.to_numpy()))

### Scalar arithmetic (both types)

Scalar operations (`+`, `-`, `*`, `/`, `abs`, `round`) work the same for univariate and multivariate series: they are applied to every value. Missing values (None/NaN) are preserved.

In [ ]:
# Univariate: scale and offset
scaled_uni = ts_uni * 1.8 + 32  # °C → °F (conceptually)
print("Univariate scaled [0]:", scaled_uni[0].value)

# Multivariate: same operation applied to all columns
scaled_multi = ts_multi * 2
print("Multivariate scaled [0]:", scaled_multi[0][1])

## 3. Quick reference

| Aspect | TimeSeries | MultivariateTimeSeries |
|--------|------------|------------------------|
| **values** | `list[float \| None]` | 2D `np.ndarray` |
| **metadata** | scalar (`name`, `unit`, ...) | lists (`names`, `units`, ...) |
| **n_columns** | — | number of columns |
| **column_names** | from `name` or `"value"` | from `names` or `value_0`, ... |
| **ts[i]** | `DataPoint(timestamp, value)` | `(timestamp, [v1, v2, ...])` |
| **to_numpy()** | 1D array | 2D array |
| **to_pandas_dataframe()** | one value column | multiple value columns |